# Lab02S01 — Demo: Coleta de Métricas CK para 1 Repositório

Este notebook demonstra o pipeline completo:
1. Clonar 1 repositório Java popular
2. Executar a ferramenta CK
3. Sumarizar as métricas CBO, DIT e LCOM
4. Gerar o arquivo CSV com os resultados

## 1. Setup e Verificações

In [ ]:
import os
import subprocess
import shutil
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".").resolve()
TOOLS_DIR = BASE_DIR / "tools"
CK_JAR = TOOLS_DIR / "ck-0.7.0-jar-with-dependencies.jar"
DATA_DIR = BASE_DIR / "data"
CLONE_DIR = DATA_DIR / "repos_clonados"
CK_OUTPUT_DIR = DATA_DIR / "ck_results"

# Verificar Java
result = subprocess.run(["java", "--version"], capture_output=True, text=True)
print(result.stdout.split('\n')[0] if result.stdout else result.stderr.split('\n')[0])

# Verificar CK JAR
if CK_JAR.exists():
    print(f"CK JAR encontrado: {CK_JAR}")
else:
    print(f"AVISO: CK JAR nao encontrado em {CK_JAR}")
    print("Baixe de: https://github.com/mauricioaniche/ck/releases")
    print(f"Coloque em: {TOOLS_DIR}/")

## 2. Baixar CK JAR (se necessário)

Execute esta célula apenas se o CK JAR ainda não estiver na pasta `tools/`.

In [ ]:
import urllib.request

CK_URL = "https://repo1.maven.org/maven2/com/github/mauricioaniche/ck/0.7.0/ck-0.7.0-jar-with-dependencies.jar"

if not CK_JAR.exists():
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Baixando CK de {CK_URL}...")
    urllib.request.urlretrieve(CK_URL, str(CK_JAR))
    print(f"CK JAR salvo em: {CK_JAR}")
else:
    print("CK JAR ja existe, pulando download.")

## 3. Clonar um repositório Java de exemplo

Usaremos o repositório `iluwatar/java-design-patterns` como exemplo.

In [ ]:
DEMO_REPO = "iluwatar/java-design-patterns"
DEMO_SAFE_NAME = DEMO_REPO.replace("/", "_")
DEMO_CLONE_PATH = CLONE_DIR / DEMO_SAFE_NAME
DEMO_CK_OUTPUT = CK_OUTPUT_DIR / DEMO_SAFE_NAME

CLONE_DIR.mkdir(parents=True, exist_ok=True)

if DEMO_CLONE_PATH.exists():
    print(f"Repositorio ja clonado em: {DEMO_CLONE_PATH}")
else:
    print(f"Clonando {DEMO_REPO} (shallow)...")
    subprocess.run(
        ["git", "clone", "--depth", "1", f"https://github.com/{DEMO_REPO}.git", str(DEMO_CLONE_PATH)],
        check=True
    )
    print("Clone concluido!")

## 4. Executar CK no repositório

In [ ]:
DEMO_CK_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Executando CK em {DEMO_CLONE_PATH}...")
print(f"Output em: {DEMO_CK_OUTPUT}")

result = subprocess.run(
    ["java", "-jar", str(CK_JAR), str(DEMO_CLONE_PATH), "false", "0", "false", str(DEMO_CK_OUTPUT) + "/"],
    capture_output=True, text=True, timeout=600
)

if result.returncode == 0:
    print("CK executado com sucesso!")
    # Listar arquivos gerados
    for f in sorted(DEMO_CK_OUTPUT.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name} ({size_kb:.1f} KB)")
else:
    print(f"ERRO: {result.stderr[:500]}")

## 5. Analisar o `class.csv` gerado pelo CK

In [ ]:
class_csv = DEMO_CK_OUTPUT / "class.csv"
df_class = pd.read_csv(class_csv)

print(f"Total de classes analisadas: {len(df_class)}")
print(f"\nColunas disponiveis: {list(df_class.columns)}")
df_class.head()

## 6. Sumarizar métricas CBO, DIT e LCOM

In [ ]:
metrics = ["cbo", "dit", "lcom"]

print(f"Repositorio: {DEMO_REPO}")
print(f"Total de classes: {len(df_class)}")
print(f"{'='*50}")

summary = {"nome": DEMO_REPO, "total_classes": len(df_class)}

for m in metrics:
    if m in df_class.columns:
        col = df_class[m].dropna()
        media = round(col.mean(), 4)
        mediana = round(col.median(), 4)
        desvio = round(col.std(), 4)
        print(f"\n{m.upper()}:")
        print(f"  Media:   {media}")
        print(f"  Mediana: {mediana}")
        print(f"  Desvio:  {desvio}")
        summary[f"{m}_media"] = media
        summary[f"{m}_mediana"] = mediana
        summary[f"{m}_desvio"] = desvio

# LOC total
if "loc" in df_class.columns:
    summary["loc_total"] = int(df_class["loc"].sum())
    print(f"\nLOC total: {summary['loc_total']:,}")

## 7. Salvar CSV com resultado da medição do repositório demo

In [ ]:
df_summary = pd.DataFrame([summary])
output_file = DATA_DIR / "metricas_ck_demo_1repo.csv"
df_summary.to_csv(output_file, index=False)

print(f"Resultado salvo em: {output_file}")
print()
df_summary

## 8. Salvar também o class.csv bruto do CK (para referência)

In [ ]:
# Copiar o class.csv bruto para a pasta data
import shutil

raw_output = DATA_DIR / f"ck_class_raw_{DEMO_SAFE_NAME}.csv"
if class_csv.exists():
    shutil.copy(class_csv, raw_output)
    print(f"class.csv bruto copiado para: {raw_output}")
    print(f"Tamanho: {raw_output.stat().st_size / 1024:.1f} KB")

## 9. Limpeza (opcional)

Remover o clone para economizar espaço em disco.

In [ ]:
# Descomente para remover o clone:
# shutil.rmtree(DEMO_CLONE_PATH, ignore_errors=True)
# print(f"Clone removido: {DEMO_CLONE_PATH}")
print("Clone mantido para referencia. Descomente a celula acima para remover.")